# 6.4 综合练习：修复并测量一个GEMM

本节依次完成三个任务：修复GEMM的读取错误、用`do_bench`计时、用`msprof`采集并读取报告。整节使用同一个算例：M=N=K=128，BM=BN=BK=64，stages=1。

<p class="course-note"><small>先尝试作答，再按需展开提示和答案。线上环境没有昇腾950，可对照页内实测结果学习；自行运行请按1.4准备环境。</small></p>

<p class="course-note"><small>在Notebook中运行时，以本章目录为工作目录，从下方准备单元开始顺序执行。完整程序和运行辅助代码可展开查看。</small></p>

In [1]:
from pathlib import Path
import re
import subprocess
import sys

Path("src").mkdir(exist_ok=True)
Path("answer").mkdir(exist_ok=True)

def run_and_show(command):
    # NPU程序在独立进程中运行；当前Notebook内核不导入torch_npu。
    result = subprocess.run(
        command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    # 只省略安装目录；数值、校验结果和报错内容保持原样。
    output = result.stdout.replace(str(Path.cwd()), ".")
    output = re.sub(r"/home/[^\s]+", "<已省略安装路径>", output)
    print(output, end="")
    result.check_returncode()
    return result


## 1. 修复GEMM中的读取错误

本题计算 **`C = A @ B.T`**，使用页首给出的固定配置。

输入A的元素全为1，B的第c行全为c+1，因此每一行输出都应满足：

```text
C[r, c] = 128 × (c + 1)
```

例如，C[0,0]应为128，C[0,64]应为8320。下面是需要检查的计算片段：

```python
row = bx // (N // BN)
col = bx % (N // BN)
b_col = (col + 1) % (N // BN)

for ki in T.Pipelined(K // BK, num_stages=1):
    T.copy(A[row * BM:(row + 1) * BM,
             ki * BK:(ki + 1) * BK], a)
    T.copy(B[b_col * BN:(b_col + 1) * BN,
             ki * BK:(ki + 1) * BK], b)
    T.gemm(a, b, acc, transpose_B=True, clear_accum=(ki == 0))

T.copy(acc, C[row * BM:(row + 1) * BM,
              col * BN:(col + 1) * BN])
```

**错误输出**

<p class="course-note"><small>日志说明：节选自错误程序的单独运行，省略编译日志和报错栈。</small></p>

```text
错误版本：不一致元素数 = 16384
错误版本：C[0,0]、C[0,64] = [8320.0, 128.0]
参考结果：C[0,0]、C[0,64] = [128.0, 8320.0]
```

请回答：哪个表达式决定了读入B的哪一组行？它与当前输出块的列范围是否对应？应修改哪一行？先保持块大小、stages和矩阵乘表达式不变。

### 逐条展开提示

<details>
<summary>提示一：先看bx=0</summary>

此时row=0、col=0，输出块覆盖C的第0～63列，它应使用B的哪64行？

</details>

<details>
<summary>提示二：用数值反推来源</summary>

错误结果8320除以K=128得到65，对应B中哪一行的值？

</details>

<details>
<summary>提示三：比较读取范围与写回范围</summary>

B的读取使用b_col，C的写回使用col。两者之间是否发生了意外偏移？

</details>



<details>
<summary>进一步核对：查看编译后的地址</summary>

不方便直接观察L1内容时，可以查看编译后错误版本的源代码。[单独复现错误的程序](answer/capstone_error.py)将这个对象保存在bad中，通过`bad.get_kernel_source()`即可取得源码，再核对B搬运语句中的源地址。

本次生成代码在bx=0、ki=0时，错误版本读取B的元素偏移为8192，正确起点应为0。因为B每行有128个元素，8192对应第64行的起点。这个观察与8320来自B[64]的推断一致。

<p class="course-note"><small>阅读提示：只需核对相关地址，无需通读整份生成代码。</small></p>

</details>

### 修复答案与完整复测

<details>
<summary>展开索引修复答案</summary>

正确的对应关系是：

```python
b_col = col
```

bx=0时应读取B[0:64, :]；bx=1时应读取B[64:128, :]。错误表达式交换了这两组行，所以前后两个输出列块的结果也随之交换。

</details>

下面的程序使用 `shift_b=False` 执行修复后的表达式，并检查题目中的输入和另外四类输入。

<p class="course-note"><small>运行说明：错误版本可单独复现；本页的可执行单元运行修复后的版本。</small></p>

In [2]:
%%writefile answer/capstone_fixed_demo.py
import tilelang
import tilelang.ascend.language as T
import torch
import torch_npu

tilelang.disable_cache()

M = N = K = 128
BM = BN = BK = 64

def make_gemm(stages=1, shift_b=False):
    @T.prim_func
    def gemm(
        A: T.Tensor((M, K), "float16"),
        B: T.Tensor((N, K), "float16"),
        C: T.Tensor((M, N), "float32"),
    ):
        with T.Kernel((M // BM) * (N // BN)) as bx:
            row = bx // (N // BN)
            col = bx % (N // BN)
            b_col = (col + 1) % (N // BN) if shift_b else col
            a = T.alloc_l1((BM, BK), "float16")
            b = T.alloc_l1((BN, BK), "float16")
            acc = T.alloc_l0c((BM, BN), "float32")
            for ki in T.Pipelined(K // BK, num_stages=stages):
                T.copy(A[row * BM:(row + 1) * BM,
                         ki * BK:(ki + 1) * BK], a)
                T.copy(B[b_col * BN:(b_col + 1) * BN,
                         ki * BK:(ki + 1) * BK], b)
                T.gemm(a, b, acc,
                       transpose_B=True, clear_accum=(ki == 0))
            T.copy(acc, C[row * BM:(row + 1) * BM,
                          col * BN:(col + 1) * BN])
    return gemm

def make_inputs(kind):
    torch.manual_seed(20260915)
    if kind == "random":
        a = (torch.randn(M, K) * 0.125).half()
        b = (torch.randn(N, K) * 0.125).half()
    elif kind == "zero":
        a = torch.zeros((M, K), dtype=torch.float16)
        b = torch.zeros((N, K), dtype=torch.float16)
    elif kind == "constant":
        a = torch.full((M, K), 0.125, dtype=torch.float16)
        b = torch.full((N, K), -0.25, dtype=torch.float16)
    else:
        a = ((torch.arange(M)[:, None] % 5 - 2).expand(M, K).float() / 16).half().contiguous()
        b = ((torch.arange(N)[:, None] % 7 - 3).expand(N, K).float() / 16).half().contiguous()
    return a, b

torch.npu.set_device(0)
a_cpu = torch.ones((M, K), dtype=torch.float16)
b_cpu = torch.arange(1, N + 1, dtype=torch.float16)[:, None].expand(N, K).contiguous()
expected = a_cpu.float() @ b_cpu.float().T
fixed = tilelang.compile(make_gemm(stages=1, shift_b=False),
                         target="ascend", out_idx=-1)
fixed_actual = fixed(a_cpu.npu(), b_cpu.npu()).cpu()
torch.testing.assert_close(fixed_actual, expected, rtol=0.002, atol=0.002)
print("修复后：C[0,0]、C[0,64] =", fixed_actual[0, [0, 64]].tolist())
fixed_checks = []
for kind in ("random", "zero", "constant", "indexed"):
    a_cpu, b_cpu = make_inputs(kind)
    actual = fixed(a_cpu.npu(), b_cpu.npu()).cpu()
    expected = a_cpu.float() @ b_cpu.float().T
    torch.testing.assert_close(actual, expected, rtol=0.002, atol=0.002)
    fixed_checks.append({"case": kind,
                         "max_abs_error": (actual - expected).abs().max().item()})
    print(f"修复后：{kind}，全部{M * N}个元素比较通过")
print("Verification passed!")


Overwriting answer/capstone_fixed_demo.py


在终端中运行，或执行下方Notebook单元：

```bash
python answer/capstone_fixed_demo.py
```

In [3]:
run_result = run_and_show([sys.executable, "-u", 'answer/capstone_fixed_demo.py'])


Loading tilelang libs from dev root: <已省略安装路径>
[W918 20:23:01.833291156 FunctionLoader.cpp:48] Warning: LD_PRELOAD detected, FunctionLoader prefers RTLD_DEFAULT for symbol resolution. (function operator())
2026-09-18 20:23:09  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:131): TileLang begins to compile kernel `gemm` with `out_idx=-1`
2026-09-18 20:23:21  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:139): TileLang completes to compile kernel `gemm`
修复后：C[0,0]、C[0,64] = [128.0, 8320.0]
修复后：random，全部16384个元素比较通过
修复后：zero，全部16384个元素比较通过
修复后：constant，全部16384个元素比较通过
修复后：indexed，全部16384个元素比较通过
Verification passed!


**先确认修复通过完整比较**

最初出错的输入和另外四类输入均通过检查后，再测量耗时。只观察C[0,0]恢复为128，仍不足以完成正确性验证。

## 2. 用do_bench测量耗时

`fixed`是修复后的GEMM经过编译得到的可调用对象。准备好设备端输入`a_npu`和`b_npu`后，调用`fixed(a_npu, b_npu)`就会在NPU上执行一次GEMM，并返回输出张量。

本题沿用第一题的输入：A全为1，B的第c行全为c+1。输入搬到NPU、算子编译完成后，再调用计时工具：

```python
from tilelang.profiler import do_bench

latency_ms = do_bench(
    __________,  # 填入一个无参数函数，调用时执行一次修复后的GEMM
    backend="msprof",
    _n_warmup=30,
    _n_repeat=100,
    cache_size=256,
)
print(f"latency_us = {__________:.5f}")  # 将毫秒换算成微秒
```

请补全两处空白，并说明 `do_bench` 的返回值表示什么。

<p class="course-note"><small>前节参照：参数含义见<a href="./06.03_measure_and_tune.ipynb">6.3性能测量</a>，本题沿用相同设置。</small></p>

<details>
<summary>展开答案</summary>

```python
from tilelang.profiler import do_bench

latency_ms = do_bench(
    lambda: fixed(a_npu, b_npu),
    backend="msprof",
    _n_warmup=30,
    _n_repeat=100,
    cache_size=256,
)
print(f"latency_ms = {latency_ms:.8f}")
print(f"latency_us = {latency_ms * 1000:.5f}")
```

`lambda: fixed(a_npu, b_npu)`定义了一个无参数函数。`do_bench`每调用它一次，就执行一次GEMM；编译和输入搬运已在这段代码之前完成。

本例由`do_bench`完成30次预热和100次采样，返回每次GEMM调用的平均耗时，单位为毫秒。乘以1000后，得到微秒数。

</details>

完整程序可展开查看，包含算子定义、编译、输入准备、结果检查和计时。

In [4]:
%%writefile answer/capstone_bench.py
import tilelang
import tilelang.ascend.language as T
import torch
import torch_npu

tilelang.disable_cache()

M = N = K = 128
BM = BN = BK = 64

def make_gemm(stages=1, shift_b=False):
    @T.prim_func
    def gemm(
        A: T.Tensor((M, K), "float16"),
        B: T.Tensor((N, K), "float16"),
        C: T.Tensor((M, N), "float32"),
    ):
        with T.Kernel((M // BM) * (N // BN)) as bx:
            row = bx // (N // BN)
            col = bx % (N // BN)
            b_col = (col + 1) % (N // BN) if shift_b else col
            a = T.alloc_l1((BM, BK), "float16")
            b = T.alloc_l1((BN, BK), "float16")
            acc = T.alloc_l0c((BM, BN), "float32")
            for ki in T.Pipelined(K // BK, num_stages=stages):
                T.copy(A[row * BM:(row + 1) * BM,
                         ki * BK:(ki + 1) * BK], a)
                T.copy(B[b_col * BN:(b_col + 1) * BN,
                         ki * BK:(ki + 1) * BK], b)
                T.gemm(a, b, acc,
                       transpose_B=True, clear_accum=(ki == 0))
            T.copy(acc, C[row * BM:(row + 1) * BM,
                          col * BN:(col + 1) * BN])
    return gemm

torch.npu.set_device(0)
fixed = tilelang.compile(make_gemm(stages=1, shift_b=False),
                         target="ascend", out_idx=-1)
a_cpu = torch.ones((M, K), dtype=torch.float16)
b_cpu = torch.arange(1, N + 1, dtype=torch.float16)[:, None].expand(N, K).contiguous()
a_npu, b_npu = a_cpu.npu(), b_cpu.npu()

c_npu = fixed(a_npu, b_npu)
actual = c_npu.cpu()
expected = a_cpu.float() @ b_cpu.float().T
torch.testing.assert_close(actual, expected, rtol=0.002, atol=0.002)
print("C[0,0], C[0,64] =", actual[0, [0, 64]].tolist())
print("Verification passed!", flush=True)

from tilelang.profiler import do_bench

latency_ms = do_bench(
    lambda: fixed(a_npu, b_npu),
    backend="msprof",
    _n_warmup=30,
    _n_repeat=100,
    cache_size=256,
)
print(f"latency_ms = {latency_ms:.8f}")
print(f"latency_us = {latency_ms * 1000:.5f}")


Overwriting answer/capstone_bench.py


在终端中运行，或执行下方Notebook单元：

```bash
python answer/capstone_bench.py
```

**从输出末尾读取平均耗时，并注意单位。**

In [5]:
run_result = run_and_show([sys.executable, "-u", 'answer/capstone_bench.py'])


Loading tilelang libs from dev root: <已省略安装路径>
[W918 20:23:28.691482672 FunctionLoader.cpp:48] Warning: LD_PRELOAD detected, FunctionLoader prefers RTLD_DEFAULT for symbol resolution. (function operator())
2026-09-18 20:23:35  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:131): TileLang begins to compile kernel `gemm` with `out_idx=-1`
2026-09-18 20:23:48  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:139): TileLang completes to compile kernel `gemm`
C[0,0], C[0,64] = [128.0, 8320.0]
Verification passed!
2026-09-18 20:23:49  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:131): TileLang begins to compile kernel `_tilelang_profiler_cache_flush` with `out_idx=[]`
2026-09-18 20:24:01  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:139): TileLang completes to compile kernel `_tilelang_profiler_cache_flush`
latency_ms = 0.00223486
latency_us = 2.23486


`Verification passed!`表示计时前的完整输出比较通过。**`latency_us`** 一行显示本次测得的平均耗时，单位为微秒。

<p class="course-note"><small>相关文件：<a href="./answer/capstone_bench.py">do_bench完整程序</a>。</small></p>

## 3. 用msprof采集并读取报告

这次使用`msprof`命令运行一个普通Python程序。程序仍然编译修复后的GEMM，准备相同的输入，再执行下面这次调用并检查结果：

```python
c_npu = fixed(a_npu, b_npu)
```

完整程序如下。它只调用一次修复后的GEMM，这次调用就是报告中要查找的目标。

In [6]:
%%writefile answer/capstone_msprof.py
import tilelang
import tilelang.ascend.language as T
import torch
import torch_npu

tilelang.disable_cache()

M = N = K = 128
BM = BN = BK = 64

def make_gemm(stages=1, shift_b=False):
    @T.prim_func
    def gemm(
        A: T.Tensor((M, K), "float16"),
        B: T.Tensor((N, K), "float16"),
        C: T.Tensor((M, N), "float32"),
    ):
        with T.Kernel((M // BM) * (N // BN)) as bx:
            row = bx // (N // BN)
            col = bx % (N // BN)
            b_col = (col + 1) % (N // BN) if shift_b else col
            a = T.alloc_l1((BM, BK), "float16")
            b = T.alloc_l1((BN, BK), "float16")
            acc = T.alloc_l0c((BM, BN), "float32")
            for ki in T.Pipelined(K // BK, num_stages=stages):
                T.copy(A[row * BM:(row + 1) * BM,
                         ki * BK:(ki + 1) * BK], a)
                T.copy(B[b_col * BN:(b_col + 1) * BN,
                         ki * BK:(ki + 1) * BK], b)
                T.gemm(a, b, acc,
                       transpose_B=True, clear_accum=(ki == 0))
            T.copy(acc, C[row * BM:(row + 1) * BM,
                          col * BN:(col + 1) * BN])
    return gemm

torch.npu.set_device(0)
fixed = tilelang.compile(make_gemm(stages=1, shift_b=False),
                         target="ascend", out_idx=-1)
a_cpu = torch.ones((M, K), dtype=torch.float16)
b_cpu = torch.arange(1, N + 1, dtype=torch.float16)[:, None].expand(N, K).contiguous()
a_npu, b_npu = a_cpu.npu(), b_cpu.npu()

c_npu = fixed(a_npu, b_npu)
actual = c_npu.cpu()
expected = a_cpu.float() @ b_cpu.float().T
torch.testing.assert_close(actual, expected, rtol=0.002, atol=0.002)
print("C[0,0], C[0,64] =", actual[0, [0, 64]].tolist())
print("Verification passed!", flush=True)


Overwriting answer/capstone_msprof.py


<p class="course-note"><small>相关文件：<a href="./answer/capstone_msprof.py">msprof完整程序</a>。</small></p>

**采集命令**

```bash
msprof --output=./prof_capstone --type=text --task-time=on \
    python answer/capstone_msprof.py
```

<p class="course-note"><small>也可以执行下方Notebook单元。采集日志可按需展开，报告生成后再读取数据。</small></p>

In [7]:
import tempfile
Path('prof_capstone').mkdir(exist_ok=True)
profile_dir = Path(tempfile.mkdtemp(prefix="run_", dir='prof_capstone'))
profile_result = run_and_show([
    "msprof", f"--output={profile_dir}", "--type=text", "--task-time=on",
    sys.executable, "-u", 'answer/capstone_msprof.py',
])


[INFO] Start profiling....
Loading tilelang libs from dev root: <已省略安装路径>
[W918 20:24:08.957066415 FunctionLoader.cpp:48] Warning: LD_PRELOAD detected, FunctionLoader prefers RTLD_DEFAULT for symbol resolution. (function operator())
2026-09-18 20:24:16  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:131): TileLang begins to compile kernel `gemm` with `out_idx=-1`
2026-09-18 20:24:28  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:139): TileLang completes to compile kernel `gemm`
C[0,0], C[0,64] = [128.0, 8320.0]
Verification passed!
[INFO] Start export data in PROF_000001_20260918202415357_00943948GPFDOEAQ.
Fri 18 Sep 2026 20:24:34 [WARNING] [MSVP] [944346] msprof_entrance.py: Msprof analysis is parsing data as root, which may cause potential system security risks.
Fri 18 Sep 2026 20:24:34 [INFO] [MSVP] [944348] msprof_common.py: Start analyzing data in "./prof_capstone/run_b95kbh31/PROF_000001_20260918202415357_00943948GPFDOEAQ/host" ...
Fri 18 Sep 2026 20:24:34 [INFO] [MSVP] [944348]

报告写入 `prof_capstone` 目录。采集完成后，在其中的 `PROF_.../mindstudio_profiler_output/` 下找到 `op_summary_*.csv` 和 `op_statistic_*.csv`。

<p class="course-note"><small>文件名说明：文件名末尾的时间戳以实际生成为准。</small></p>

### 对照报告，读出耗时与次数

先带着下面两个问题查看报告：

1. 这次GEMM调用的耗时是多少？从哪个文件、哪一列读出？
2. `Count=1`表示什么？为什么本例的最小、平均和最大耗时相同？

下方两张表直接读取本次CSV，显示目标kernel的调用明细和汇总结果。读取代码可展开查看。

In [8]:
import csv
import shutil
from IPython.display import Markdown, display

def load_target(pattern, name_field):
    paths = list(profile_dir.rglob(pattern))
    if len(paths) != 1:
        raise RuntimeError(f"本次目录中应有一份{pattern}，实际找到{len(paths)}份")
    with paths[0].open(encoding="utf-8-sig", newline="") as file:
        rows = [row for row in csv.DictReader(file) if row[name_field] == 'gemm_kernel']
    if not rows:
        raise RuntimeError("本次报告中没有找到目标算子")
    return paths[0], rows

summary_path, summary_rows = load_target("op_summary_*.csv", "Op Name")
statistic_path, statistic_rows = load_target("op_statistic_*.csv", "OP Type")

def show_table(title, rows, columns):
    lines = ["**" + title + "**", "", "| " + " | ".join(columns) + " |",
             "| " + " | ".join("---" for _ in columns) + " |"]
    lines += ["| " + " | ".join(row[column].strip() for column in columns) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

print("本次明细报告：", summary_path.relative_to(Path.cwd()))
print("本次汇总报告：", statistic_path.relative_to(Path.cwd()))
show_table("单次调用", summary_rows, ['Op Name', 'Task Type', 'Task Duration(us)', 'Block Num'])
show_table("汇总结果", statistic_rows, ['OP Type', 'Count', 'Total Time(us)', 'Min Time(us)', 'Avg Time(us)', 'Max Time(us)'])

# 保存同一批报告，便于离线打开；表格内容始终取自上面的本次采集目录。
sample_dir = Path('data/capstone_msprof_sample')
sample_dir.mkdir(parents=True, exist_ok=True)
shutil.copyfile(summary_path, sample_dir / "op_summary_current.csv")
_ = shutil.copyfile(statistic_path, sample_dir / "op_statistic_current.csv")


本次明细报告： prof_capstone/run_b95kbh31/PROF_000001_20260918202415357_00943948GPFDOEAQ/mindstudio_profiler_output/op_summary_20260918202437.csv
本次汇总报告： prof_capstone/run_b95kbh31/PROF_000001_20260918202415357_00943948GPFDOEAQ/mindstudio_profiler_output/op_statistic_20260918202437.csv


**单次调用**

| Op Name | Task Type | Task Duration(us) | Block Num |
| --- | --- | --- | --- |
| gemm_kernel | AI_CORE | 5.278 | 4 |

**汇总结果**

| OP Type | Count | Total Time(us) | Min Time(us) | Avg Time(us) | Max Time(us) |
| --- | --- | --- | --- | --- | --- |
| gemm_kernel | 1 | 5.278 | 5.278 | 5.278 | 5.278 |

<details>
<summary>展开报告阅读答案</summary>

本次调用的耗时直接读取上方明细表的`Task Duration(us)`列，单位为微秒；表格来自本次`op_summary_*.csv`。该列记录设备端任务的耗时；不要把终端中编译和运行整个Python程序所用的时间填到这里。

`Count=1`表示汇总中只有一次`gemm_kernel`任务。因此，本例的最小值、平均值和最大值都等于这一次调用的耗时。

</details>

**比较前先看测量条件：** 这份报告记录未额外预热的**单次调用**；上一题的 `latency_us` 是预热并多次采样后的**平均耗时**。两次采集条件不同，数值也会不同。

<p class="course-note"><small>相关文件：<a href="./data/capstone_msprof_sample/op_summary_current.csv">调用明细CSV</a> · <a href="./data/capstone_msprof_sample/op_statistic_current.csv">汇总CSV</a>，可查看完整字段。</small></p>

## 完成检查

完成本节后，检查自己能否独立完成以下操作：

- 根据错误输出找出读错的B块，修改索引，并比较完整输出。
- 把一次算子调用写成`do_bench`接收的函数，读出返回值并换算成微秒。
- 用`msprof`运行算子程序，在CSV中找到目标kernel的耗时和调用次数。

[章节导航](README.md) · [上一节](06.03_measure_and_tune.ipynb)